# Week 11 · Notebook 1 — A Working RAG Pipeline
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Build the full pipeline over a tiny document set:
**load → chunk → embed → store (ChromaDB) → retrieve → ground → generate (Claude + Gemini) → cite.**

> Runs in Google Colab. Embeddings are **local Hugging Face** (no OpenAI). Generation cells **degrade gracefully** if a key/package is missing.

## 0. Install

In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters \
    chromadb sentence-transformers anthropic google-generativeai

## 1. Load API keys safely (optional — retrieval works without them)

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except Exception:
    pass
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 2. Our document set (a course handbook)

In [ ]:
COURSE_HANDBOOK = '''
Attendance policy: This is an online asynchronous course. Modules open Monday
and the weekly lab is due Sunday at 11:59 PM Pacific. You may miss deadlines
twice with no penalty using the two automatic 48-hour extensions.

AI-use policy: Students may use AI assistants such as Claude and Gemini on
assignments but must understand and disclose their use with an AI Use note.
Exams, including the midterm and final, are AI-restricted.

Grading: Labs and assignments are 50 percent, the midterm is 20 percent, and
the final project is 30 percent. The final project is due December 19.

Office hours: Tuesday 5 to 6 PM and Saturday 10 to 11 AM Pacific on Zoom.
Email the instructor with the subject prefix CSCI250 for a reply within 48 hours.
'''
print(COURSE_HANDBOOK)

## 3. Chunk the document
Split into small overlapping passages. Inspect the chunks — retrieval can only find what chunking produced.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=240, chunk_overlap=40)
chunks = splitter.split_text(COURSE_HANDBOOK)
print(len(chunks), 'chunks')
for i, c in enumerate(chunks):
    print(f'--- [{i}] ---')
    print(c.strip())

## 4. Embed & store in ChromaDB (with metadata for citations)
We attach a `source` and `chunk` number to every chunk — that metadata is what lets us cite later. Chroma auto-embeds with a local sentence-transformers model.

In [ ]:
import chromadb
client = chromadb.Client()
try:
    client.delete_collection('handbook')
except Exception:
    pass
col = client.create_collection('handbook')
col.add(
    documents=[c.strip() for c in chunks],
    metadatas=[{'source': 'handbook.txt', 'chunk': i} for i in range(len(chunks))],
    ids=[f'c{i}' for i in range(len(chunks))],
)
print('stored', col.count(), 'chunks')

## 5. Retrieve the most relevant chunks

In [ ]:
def retrieve(question, k=3):
    res = col.query(query_texts=[question], n_results=k)
    return res['documents'][0], res['metadatas'][0]

QUESTION = 'How is the final project weighted and when is it due?'
retrieved, sources = retrieve(QUESTION, k=3)
for d, m in zip(retrieved, sources):
    print(f"[{m['chunk']}] {d}\n")

## 6. Ground the prompt
Tell the model to answer ONLY from the retrieved context, to say "I don't know" otherwise, and to cite sources by number.

In [ ]:
def build_prompt(question, retrieved):
    context = '\n\n'.join(f'[{i+1}] {c}' for i, c in enumerate(retrieved))
    return (
        'Answer the question using ONLY the context below.\n'
        'If the answer is not in the context, say '
        '"I don\'t know based on the provided documents."\n'
        'Cite the sources you used by their [number].\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {question}\nAnswer:'
    )

prompt = build_prompt(QUESTION, retrieved)
print(prompt)

## 7. Generate with Claude

In [ ]:
def answer_with_claude(prompt):
    try:
        import anthropic
        msg = anthropic.Anthropic().messages.create(
            model='claude-sonnet-4-6', max_tokens=400,
            messages=[{'role': 'user', 'content': prompt}])
        return msg.content[0].text
    except Exception as e:
        return f'[Claude unavailable: {e}]'

print('CLAUDE:\n', answer_with_claude(prompt))

## 8. Generate with Gemini (same context)

In [ ]:
def answer_with_gemini(prompt):
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        model = genai.GenerativeModel('gemini-2.5-flash')
        return model.generate_content(prompt).text
    except Exception as e:
        return f'[Gemini unavailable: {e}]'

print('GEMINI:\n', answer_with_gemini(prompt))

## 9. Cite the sources
Because each chunk carried metadata, we can show exactly what the answer was grounded in.

In [ ]:
print('Sources:')
for i, m in enumerate(sources, 1):
    print(f"  [{i}] {m['source']} (chunk {m['chunk']})")

## 10. One function, the whole pipeline
Wrap it up so you can ask anything. Reuse this in your Final Project.

In [ ]:
def rag(question, k=3, model='claude'):
    retrieved, sources = retrieve(question, k=k)
    prompt = build_prompt(question, retrieved)
    gen = answer_with_claude if model == 'claude' else answer_with_gemini
    answer = gen(prompt)
    cites = '\n'.join(f"  [{i+1}] {m['source']} (chunk {m['chunk']})"
                       for i, m in enumerate(sources))
    return f'{answer}\n\nSources:\n{cites}'

print(rag('What is the AI-use policy on exams?', model='claude'))